In [0]:

USE main_dev;

-- Enforce Task 2 Idempotency Rules: 
-- CREATE OR REPLACE TABLE ensures that running this file multiple times 
-- completely overwrites the destination target without appending duplicate records.
CREATE OR REPLACE TABLE main_dev.daily_account_summary AS

WITH filtered_transactions AS (
    SELECT 
        account_id,
        -- Truncate string timestamp to a clean UTC Date object
        CAST(TO_TIMESTAMP(transaction_date) AS DATE) AS transaction_date,
        -- Cast string amount to Decimal for precise financial calculations
        CAST(amount AS DECIMAL(18, 4)) AS amount_numeric,
        transaction_type,
        merchant_name,
        merchant_category,
        currency
    FROM main_dev.bronze_transactions
    -- Filter out quarantined data implicitly and keep only completed records
    WHERE status = 'completed'
),

daily_base_metrics AS (
    SELECT
        account_id,
        transaction_date,
        COALESCE(SUM(CASE WHEN transaction_type = 'debit' THEN amount_numeric ELSE 0 END), 0) AS total_debit_amount,
        COALESCE(SUM(CASE WHEN transaction_type = 'credit' THEN amount_numeric ELSE 0 END), 0) AS total_credit_amount,
        COUNT(1) AS transaction_count,
        COUNT(DISTINCT merchant_name) AS distinct_merchants,
        -- Collect distinct currencies into an alphabetized comma-separated string
        ARRAY_JOIN(ARRAY_SORT(ARRAY_DISTINCT(COLLECT_LIST(currency))), ', ') AS currencies
    FROM filtered_transactions
    GROUP BY account_id, transaction_date
),

category_spend_ranking AS (
    SELECT 
        account_id,
        transaction_date,
        merchant_category,
        ROW_NUMBER() OVER (
            PARTITION BY account_id, transaction_date 
            ORDER BY SUM(amount_numeric) DESC, merchant_category ASC
        ) AS category_rank
    FROM filtered_transactions
    GROUP BY account_id, transaction_date, merchant_category
),

top_category_filtered AS (
    SELECT 
        account_id,
        transaction_date,
        merchant_category AS top_category
    FROM category_spend_ranking
    WHERE category_rank = 1
)

SELECT
    m.account_id,
    m.transaction_date,
    CAST(m.total_debit_amount AS DECIMAL(18, 2)) AS total_debit_amount,
    CAST(m.total_credit_amount AS DECIMAL(18, 2)) AS total_credit_amount,
    CAST((m.total_credit_amount - m.total_debit_amount) AS DECIMAL(18, 2)) AS net_amount,
    m.transaction_count,
    m.distinct_merchants,
    t.top_category,
    m.currencies,
    CURRENT_TIMESTAMP() AS updated_at
FROM daily_base_metrics m
LEFT JOIN top_category_filtered t 
    ON m.account_id = t.account_id 
    AND m.transaction_date = t.transaction_date;


In [0]:
%python
import os

# 1. Map target directory paths (matching your user folder mapping layout)
TARGET_DIR = "/Workspace/Users/sourav.dw91@gmail.com/senior-de-assignment/outputs"
TARGET_FILE = f"{TARGET_DIR}/daily_account_summary_sample.csv"

# 2. Safely generate the nested 'outputs' directory using Python's native tool layer
# Python's built-in os.makedirs works within WSFS, whereas shell 'mkdir' is blocked
if not os.path.exists(TARGET_DIR):
    os.makedirs(TARGET_DIR, exist_ok=True)
    print(f"Verified target destination path parameters: {TARGET_DIR}")

# 3. Pull query results down directly into driver space as a clean string buffer
print("Extracting data metrics directly from SQL summary warehouse matrix...")
sample_df = spark.sql("SELECT * FROM main_dev.daily_account_summary LIMIT 10").toPandas()
csv_string_buffer = sample_df.to_csv(index=False)

# 4. Stream data directly to the file inside the Git folder using an active workspace stream
# This authenticates via your user profile, bypassing the 'DIRECTORY_PROTECTED' block
print("Streaming sample metrics down to repository files tracking layout...")
with open(TARGET_FILE, "w", encoding="utf-8") as f:
    f.write(csv_string_buffer)

# 5. Operational Quality Verification Check
if os.path.exists(TARGET_FILE) and os.path.getsize(TARGET_FILE) > 0:
    print(f"SUCCESS: Successfully generated daily_account_summary_sample.csv inside your outputs folder!")
else:
    raise IOError("Operational state verification failed. Data metrics payload stream interrupted.")
